# Building a Phishing Email Classifier — From Scratch to Pydantic AI + Gradio

**A hands-on walkthrough: Manual Detection → Pydantic AI → Tools → Streaming → Gradio UI**

---

## Agenda

| # | Topic | What you'll learn |
|---|-------|-------------------|
| 1 | **Detector from Scratch** | Build a phishing detector using plain Python + LiteLLM |
| 2 | **Pydantic AI Classifier** | Same detector, rebuilt with Pydantic AI — structured threat reports |
| 3 | **Adding Tools** | Register URL extraction, domain checking, and header analysis tools |
| 4 | **Streaming Agent** | Stream phishing analysis tokens in real-time using `run_stream()` |
| 5 | **Gradio UI** | Wrap the streaming agent in a live phishing scanner interface |

## Setup

In [ ]:
!pip install -q litellm requests pydantic-ai gradio python-dotenv

In [ ]:
import os

os.environ["GROQ_API_KEY"] = "<your-groq-api-key>"  # console.groq.com

print("GROQ_API_KEY:", "set ✓" if os.getenv("GROQ_API_KEY") else "MISSING ✗")

---

# Part 1 — Phishing Detector from Scratch

## What is a Phishing Email?

A **phishing email** is a fraudulent message designed to:

- **Steal** credentials (username, password, OTP)
- **Trick** victims into clicking malicious links
- **Impersonate** trusted brands or organizations
- **Create urgency** to bypass critical thinking

```
Email Input
    │
    ▼
LLM (litellm)  ──►  JSON response
    │                   │
    │          ┌────────┴────────────┐
    │      is_phishing?            no
    │          │                    │
    ▼          ▼                    ▼
    Extract red flags         Safe email
          │
          ▼
    Generate threat report
```

### Protocol
We force the LLM to always reply with **structured JSON**:

```json
{
  "is_phishing": true,
  "confidence_score": 0.92,
  "severity": "critical",
  "red_flags": ["Spoofed domain paypa1.com", "Urgent language"],
  "recommended_action": "Delete immediately and report"
}
```

No framework magic — just Python + an LLM API.

![Cybersecurity GIF](https://media.giphy.com/media/v1.Y2lkPTc5MGI3NjExbDhqanRvZjBvZDBiYm9pc2Z6eTRnMGFydmJqYnI1aWEwY3VsMWZ0eCZlcD12MV9naWZzX3NlYXJjaCZjdD1n/077i6AULCXc0FKTj9s/giphy.gif)

## Step 1 — Sample Phishing Email

Let's start with a real phishing example (PayPal spoofing).

In [ ]:
SAMPLE_PHISHING_EMAIL = """
From: security@paypa1-alerts.com
Subject: URGENT: Your account has been limited!

Dear Valued Customer,

Your PayPal account access is limited. Verify immediately at 
http://paypa1-alerts.com/verify?token=abc123 or your account 
will be permanently closed within 24 hours.

Thank you for your immediate attention.
PayPal Security Team
"""

print("Sample phishing email loaded ✓")
print(SAMPLE_PHISHING_EMAIL)

## Step 2 — System Prompt + LLM Caller

The system prompt tells the LLM **exactly what to check**.  
`call_llm()` sends the email and parses the JSON reply.

In [ ]:
import json
from litellm import completion

MODEL = "groq/llama-3.3-70b-versatile"

SYSTEM_PROMPT = """
You are a senior cybersecurity analyst specialized in phishing detection.

When analyzing an email always check:
1. Sender domain vs claimed brand (e.g. paypa1.com vs paypal.com)
2. Urgency or fear language ("account suspended", "act now")
3. Suspicious or obfuscated URLs
4. Grammar mistakes and inconsistencies
5. Mismatched Reply-To headers
6. Requests for credentials, OTP, or payment

You MUST respond with valid JSON only. No extra text, no markdown.

Response format:
{
  "is_phishing": true/false,
  "confidence_score": 0.0-1.0,
  "attack_type": "credential_harvesting" | "malware_link" | "business_email_compromise" | "fake_invoice" | "none",
  "severity": "low" | "medium" | "high" | "critical",
  "red_flags": ["list of specific observations"],
  "spoofed_brand": "brand name or null",
  "recommended_action": "what to do",
  "explanation": "detailed reasoning"
}
"""


def call_llm(messages: list[dict]) -> dict:
    """Send messages to the LLM and return its response as a Python dict."""
    response = completion(
        model=MODEL,
        messages=messages,
        response_format={"type": "json_object"},
    )
    raw_text = response.choices[0].message.content
    return json.loads(raw_text)


print("LLM caller ready ✓")

## Step 3 — Analyze the Email

Send the phishing email to the LLM and get a structured threat report.

In [ ]:
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": f"Analyze this email:\n{SAMPLE_PHISHING_EMAIL}"},
]

result = call_llm(messages)

print("="*60)
print("THREAT ANALYSIS REPORT")
print("="*60)
print(json.dumps(result, indent=2))
print("="*60)

### What we just built

| Component | Code |
|-----------|------|
| LLM call | `litellm.completion()` with `response_format={"type": "json_object"}` |
| System prompt | Detailed phishing detection criteria + JSON schema |
| Response parsing | `json.loads(response.text)` |

**Total: ~40 lines of Python.** No magic, fully transparent.

![Security](https://media.giphy.com/media/v1.Y2lkPTc5MGI3NjExZm93emhpcXRob3g5NnRveGpoYWFqaG1wZWNkNGVkaW93ZGxoeWh5ZSZlcD12MV9naWZzX3NlYXJjaCZjdD1n/26tPnAAJxXTvpLwJy/giphy.gif)

---

# Part 2 — Pydantic AI Classifier

## Why Pydantic AI?

The from-scratch detector works, but there's boilerplate:

- Writing the JSON schema in the system prompt
- Manually parsing and validating the response
- No type safety for the output

**Pydantic AI** handles all of that. You focus on the *what*, not the *how*.

```
# Before (from scratch)              # After (Pydantic AI)
result = call_llm(messages)          result = agent.run_sync(prompt)
  ├─ manual JSON prompt                # ← auto schema generation
  ├─ json.loads()                      # ← auto validation
  └─ dict (no types)                   # ← typed ThreatReport object
```

![agent](https://media.giphy.com/media/v1.Y2lkPTc5MGI3NjExYXhsdTQ1eWZnOG1ndnE5YzU0cHIwbDI3OWxhYXJveGk3M2dlenFoZCZlcD12MV9naWZzX3NlYXJjaCZjdD1n/xT0GqsL0hdM0ueOSzu/giphy.gif)

## 2a — Plain Text Analysis

The simplest Pydantic AI agent: one model, one system prompt, one call.

In [ ]:
from pydantic_ai import Agent

basic_agent = Agent(
    model="groq:llama-3.3-70b-versatile",
    system_prompt="You are a cybersecurity expert. Analyze emails for phishing threats.",
)

# Jupyter already has a running event loop — use `await` directly
result = await basic_agent.run(f"Is this email phishing?\n{SAMPLE_PHISHING_EMAIL}")
print(result.output)

## 2b — Structured Output (Pydantic Models)

Pass a **Pydantic model** as `output_type` and Pydantic AI:
1. Generates a JSON schema from your model
2. Sends it to the LLM to constrain its output
3. Parses and validates the response back into your model

No prompt engineering. No `json.loads()`. No `try/except`.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal
from pydantic_ai import Agent


class ThreatReport(BaseModel):
    is_phishing: bool
    confidence_score: float = Field(..., ge=0.0, le=1.0)
    attack_type: Literal[
        "credential_harvesting",
        "malware_link",
        "business_email_compromise",
        "fake_invoice",
        "none"
    ]
    severity: Literal["low", "medium", "high", "critical"]
    red_flags: list[str]
    spoofed_brand: str | None
    recommended_action: str
    explanation: str


structured_agent = Agent(
    model="groq:llama-3.3-70b-versatile",
    output_type=ThreatReport,
    system_prompt="""
    You are a senior cybersecurity analyst. Analyze the email for phishing.
    Check sender domain, urgency language, suspicious URLs, and grammar.
    Never set confidence above 0.95. Be precise and conservative.
    """,
)

# Jupyter already has a running event loop — use `await` directly
result = await structured_agent.run(SAMPLE_PHISHING_EMAIL)
report: ThreatReport = result.output

print(report.model_dump_json(indent=2))

The result is a real Python object — `report.severity` is a validated `Literal`, `report.confidence_score` is a `float` between 0-1.  
If the model returns an invalid shape, Pydantic raises a `ValidationError` — fail fast with a clear message.

---

# Part 3 — Adding Tools

## How tools work in Pydantic AI

`@agent.tool_plain` turns a plain function into a tool the LLM can call.  
Pydantic AI reads the **type annotations** to build the JSON schema automatically — no manual schema writing.

```
@agent.tool_plain
def extract_urls(email_body: str) -> list[str]:
    ...                      ↑
             Pydantic AI reads this → generates schema
```

When the LLM wants to call `extract_urls`, Pydantic AI:
1. Intercepts the tool-call request
2. Validates the arguments against the schema
3. Calls your function
4. Feeds the result back to the LLM
5. Repeats until the LLM produces a final answer

![Tools](https://media.giphy.com/media/v1.Y2lkPTc5MGI3NjExcHVwd3R0c2o2aHI1OWVzdGw4NnA3aHQzMmtyZGR2YzBmdGt1czgxZSZlcD12MV9naWZzX3NlYXJjaCZjdD1n/xTiTnxpQ3ghPiB2Hp6/giphy.gif)

In [ ]:
import re
import requests
from pydantic_ai import Agent


# Use a tool-calling optimized model
phishing_agent = Agent(
    model="groq:llama-3.3-70b-versatile",
    output_type=ThreatReport,
    system_prompt="""
    You are a senior cybersecurity analyst with 10+ years of 
    phishing detection experience.

    When analyzing an email always check:
    1. Sender domain vs claimed brand
    2. Urgency or fear language
    3. Suspicious or obfuscated URLs
    4. Grammar mistakes and inconsistencies
    5. Mismatched Reply-To headers
    6. Requests for credentials, OTP, or payment

    SEVERITY GUIDE:
    - critical: spoofed major brand + credential harvesting
    - high: malware link or BEC attempt
    - medium: suspicious signals but unclear intent
    - low: mild spam, unlikely phishing

    Use the available tools to extract URLs and check domains.
    Always populate red_flags with specific observations.
    Never set confidence above 0.95.
    """,
)


@phishing_agent.tool_plain
def extract_urls(email_body: str) -> list[str]:
    """Extract all http/https URLs from the email body using regex."""
    url_pattern = r'https?://[^\s<>"]+|www\.[^\s<>"]+'
    urls = re.findall(url_pattern, email_body)
    return urls if urls else []


@phishing_agent.tool_plain
def check_domain_suspicion(domain: str) -> dict:
    """Check if domain contains typosquatting patterns (paypa1, amaz0n, etc)."""
    domain_lower = domain.lower()
    suspicious_patterns = {
        'paypal': ['paypa1', 'paypai', 'paypa'],
        'amazon': ['amaz0n', 'arnazon', 'amazom'],
        'google': ['g00gle', 'gooogle', 'googie'],
        'microsoft': ['micros0ft', 'rnicrosoft', 'microsof'],
        'apple': ['app1e', 'appl', 'appie'],
        'facebook': ['faceb00k', 'facebok', 'faceboook'],
        'netflix': ['netfl1x', 'netfllx', 'netfiix'],
        'bank': ['bankk', 'bnak', 'b4nk']
    }
    for brand, patterns in suspicious_patterns.items():
        for pattern in patterns:
            if pattern in domain_lower:
                return {"domain": domain, "is_suspicious": True,
                        "reason": f"Typosquatting of {brand} ('{pattern}')"}
    return {"domain": domain, "is_suspicious": False,
            "reason": "No typosquatting detected"}


@phishing_agent.tool_plain
def analyze_header_mismatch(from_header: str, reply_to: str) -> dict:
    """Check if From and Reply-To domains are different (common in phishing)."""
    def get_domain(email: str) -> str:
        match = re.search(r'@([\w.-]+)', email)
        return match.group(1) if match else ""
    from_d = get_domain(from_header)
    reply_d = get_domain(reply_to)
    return {"mismatch": from_d != reply_d and bool(from_d) and bool(reply_d),
            "from": from_d, "reply_to": reply_d}


print("Phishing agent with tools ready ✓")

In [ ]:
# Jupyter already has a running event loop — use `await` directly
result = await phishing_agent.run(SAMPLE_PHISHING_EMAIL)
report: ThreatReport = result.output

print("="*60)
print("PHISHING THREAT REPORT")
print("="*60)
print(f"🚨 Phishing: {report.is_phishing}")
print(f"⚠️  Severity: {report.severity.upper()}")
print(f"📊 Confidence: {report.confidence_score:.1%}")
print(f"🎯 Attack Type: {report.attack_type}")
print(f"🏷️  Spoofed Brand: {report.spoofed_brand}")
print(f"\n🚩 Red Flags:")
for flag in report.red_flags:
    print(f"   • {flag}")
print(f"\n✅ Action: {report.recommended_action}")

In [ ]:
# Test with a legitimate email
LEGIT_EMAIL = """
From: newsletter@github.com
Subject: Your GitHub digest for this week

Hi there,

Here are the trending repositories this week on GitHub:
1. awesome-ai-tools - A curated list of AI tools
2. machine-learning-course - Free ML course materials

Visit github.com/trending to explore more.

Happy coding!
The GitHub Team
"""

result = await phishing_agent.run(LEGIT_EMAIL)
report = result.output
print(f"Phishing: {report.is_phishing} | Severity: {report.severity} | Confidence: {report.confidence_score:.1%}")

---

# Part 4 — Streaming Agent

## Real-time token streaming with `run_stream()`

By default, `agent.run()` waits for the full response before returning.  
With `run_stream()` you receive tokens as they arrive — ideal for long analysis reports.

```
agent.run()                      agent.run_stream()
─────────────────                ──────────────────────────────────────
[LLM thinking……………]             [LLM] "This" → " email" → " is" → " …"
          │                                  │
          ▼                                  ▼
  return full text              yield new tokens on each chunk
```

> **Note**: For streaming we use a plain-text agent (no `output_type`) since  
> structured output + streaming is not always supported by all providers.

We create a **`streaming_agent`** without tools or structured output for pure streaming.

In [ ]:
from pydantic_ai import Agent

# Dedicated streaming agent — no tools, no structured output.
# Groq raises an error when run_stream() is combined with tool-calling,
# so we keep this agent clean for pure streaming.
streaming_agent = Agent(
    model="groq:llama-3.3-70b-versatile",
    system_prompt=(
        "You are a cybersecurity expert. Analyze emails for phishing threats. "
        "Give clear, concise analysis with red flags, severity, and recommended actions."
    ),
)

print("Streaming agent ready ✓")

In [ ]:
# delta=True → each chunk contains only the NEW tokens (incremental).
# We print them with end="" so they append in place.

async with streaming_agent.run_stream(
    f"Analyze this email for phishing:\n{SAMPLE_PHISHING_EMAIL}"
) as result:
    async for text in result.stream_text(delta=True):
        print(text, end="", flush=True)

print()  # final newline

---

# Part 5 — Gradio Chat UI (Streaming)

## Wrapping the streaming agent in a Gradio UI

Gradio's `ChatInterface` accepts an **`async` generator** to stream tokens live.  
We pass each growing chunk as a `yield` — Gradio updates the chat bubble on every tick.

> **Note on Groq + tools + streaming**: Groq raises an API error when you
> combine `run_stream()` with tool-calling.  
> So in the Gradio UI we use the `streaming_agent` **without** tools.

```
User pastes email
    │
    ▼
Gradio calls stream_response(message, history)
    │
    ▼
streaming_agent.run_stream(message)     ← no tool calls, pure streaming
    │
    ├─ chunk 1 → yield partial_text  →  Gradio updates bubble
    ├─ chunk 2 → yield partial_text  →  Gradio updates bubble
    └─ done    → final text displayed
```

In [ ]:
import gradio as gr


async def stream_response(user_message: str, _history):
    """Async generator — yields the growing text on every streamed chunk."""
    partial = ""
    async with streaming_agent.run_stream(user_message) as result:
        async for text in result.stream_text(delta=False):
            partial = text
            yield partial


def build_ui():
    with gr.Blocks(title="PhishGuard — AI Phishing Scanner") as demo:
        gr.Markdown("## 🛡️ PhishGuard — AI Phishing Email Scanner")
        gr.Markdown(
            "Paste a suspicious email below. "
            "The AI will analyze it for phishing threats in real-time."
        )
        gr.ChatInterface(
            fn=stream_response,
            examples=[
                "From: security@paypa1-alerts.com\nSubject: URGENT: Your account has been limited!\n\nDear Customer, verify at http://paypa1-alerts.com/verify?token=abc123 or your account will be closed in 24 hours.",
                "From: ceo@company-corp.net\nSubject: Urgent Wire Transfer\n\nI need you to process a $47,000 wire transfer immediately. Do not discuss with anyone. Reply to: finance@gmail.com",
                "From: newsletter@github.com\nSubject: Your weekly digest\n\nHere are trending repos this week. Visit github.com/trending to explore more.",
            ],
        )
    return demo


build_ui().launch()

### How the streaming Gradio integration works

```
User pastes email
    │
    ▼
Gradio calls stream_response(message, history)   ← async generator
    │
    ▼
async with streaming_agent.run_stream(message) as result:
    async for text in result.stream_text(delta=False):
        yield text          ← Gradio re-renders the bubble on every yield
```

Key points:
- `stream_response` is an **async generator** (`yield` instead of `return`) — Gradio detects this automatically and streams the output.
- `delta=False` sends the **full text so far** on every chunk, so Gradio always has the complete string to display.
- The `streaming_agent` has **no tools attached** — Groq's API does not support streaming + tool-calling simultaneously.

---

# Summary

## The Progression

```
1. Detector from Scratch
   litellm + manual JSON protocol
   → Full visibility, ~40 lines, great for learning
         │
         ▼
2. Pydantic AI — Structured Output
   Agent(model=..., output_type=ThreatReport)
   → Typed threat reports, automatic validation
         │
         ▼
3. Pydantic AI — With Tools
   @agent.tool_plain → URL extraction, domain checking
   → Enhanced detection, ~15 lines per tool
         │
         ▼
4. Streaming Agent
   agent.run_stream() + stream_text(delta=True)
   → Tokens arrive in real-time; reuses existing agent
         │
         ▼
5. Gradio UI (Streaming)
   async generator + gr.ChatInterface
   → Live phishing scanner in the browser
```

## Quick Reference

| Pattern | API | Use when |
|---------|-----|----------|
| From scratch | `litellm.completion()` + manual JSON | Learning, full control |
| Unstructured output | `await agent.run(prompt)` | Simple Q&A |
| Structured output | `Agent(output_type=ThreatReport)` | Typed data, downstream processing |
| Tools | `@agent.tool_plain` | URL extraction, domain checking |
| Streaming | `agent.run_stream()` + `stream_text()` | Long analysis, notebooks |
| Streaming + UI | async generator + `gr.ChatInterface` | Live phishing scanner |

## Groq Streaming Limitation

| Combination | Works? |
|-------------|--------|
| `run_stream()` — no tools | ✓ Full streaming |
| `run_stream()` — with tools | ✗ Groq API error |
| `agent.run()` — with tools | ✓ Full tool loop, response arrives at once |